Credit Scoring - Estimating the probability that a customer will default
Fraud Detection - Estimating the probability that a transaction is fraudulent.
Machine Learning - Automated probability estimation

In [8]:
#Let's build a richer scoring dataset
import pandas as pd
import numpy as np

In [9]:
customers = {

    "customer_id": range(1,16),

    "monthly_income":[
        50000,
        70000,
        85000,
        100000,
        120000,
        140000,
        160000,
        180000,
        200000,
        220000,
        250000,
        280000,
        320000,
        400000,
        500000
    ],

    "days_in_arrears":[
        0,
        0,
        5,
        0,
        10,
        15,
        0,
        20,
        30,
        45,
        60,
        90,
        0,
        0,
        0
    ]
}

df = pd.DataFrame(customers)

df

,customer_id,monthly_income,days_in_arrears
0,1,50000,0
1,2,70000,0
2,3,85000,5
3,4,100000,0
4,5,120000,10
5,6,140000,15
6,7,160000,0
7,8,180000,20
8,9,200000,30
9,10,220000,45


In [10]:
#mean
df["monthly_income"].mean()


np.float64(205000.0)

In [11]:
#median
df["monthly_income"].median()

np.float64(180000.0)

Our business question: Wich represents the customer base better:
This will teach us 1: central tendency. 2. outlier influence and 3. portfolio representation


In [12]:
#Standard Deviation - how much the data is spread out from the mean
#small std means customers are similar, large std means customers vary significantly
df["monthly_income"].std()

np.float64(127601.50021509487)

In [15]:
#customers far from the mean are outliers, and they require investigation
#Z-score is a way to identify outliers, it measures how many standard deviations a data point is from the mean
#Formula: Z = (X - mean) / std
#where X is the data point, mean is the average of the data, and std is the standard deviation of the data
df["z_score"] = (df["monthly_income"] - df["monthly_income"].mean ()) / df["monthly_income"].std()
df

,customer_id,monthly_income,days_in_arrears,z_score
0,1,50000,0,-1.214719
1,2,70000,0,-1.057981
2,3,85000,5,-0.940428
3,4,100000,0,-0.822874
4,5,120000,10,-0.666136
5,6,140000,15,-0.509398
6,7,160000,0,-0.352660
7,8,180000,20,-0.195922
8,9,200000,30,-0.039184
9,10,220000,45,0.117553


Interpretation
if z=0, customer is average
if z=2, customer is unusually high
if z=-2, customer is unusually low
Z-scores are used in:
- fraud detection
- anomaly detection
- monitoring

In [16]:
#risk buckets - lets convert arrears into risk 
def risk_bucket(days):

    if days == 0:
        return "Current"

    elif days <= 30:
        return "Watchlist"

    elif days <= 60:
        return "High Risk"

    else:
        return "Default"


In [18]:
#apply risk buckets
df["risk_bucket"] = df[
    "days_in_arrears"
].apply(risk_bucket)
df

,customer_id,monthly_income,days_in_arrears,z_score,risk_bucket
0,1,50000,0,-1.214719,Current
1,2,70000,0,-1.057981,Current
2,3,85000,5,-0.940428,Watchlist
3,4,100000,0,-0.822874,Current
4,5,120000,10,-0.666136,Watchlist
5,6,140000,15,-0.509398,Watchlist
6,7,160000,0,-0.352660,Current
7,8,180000,20,-0.195922,Watchlist
8,9,200000,30,-0.039184,Watchlist
9,10,220000,45,0.117553,High Risk


In [19]:
#let's now build a simple credit score
def credit_score(row):

    score = 500

    if row["monthly_income"] > 200000:
        score += 100

    if row["days_in_arrears"] == 0:
        score += 150

    elif row["days_in_arrears"] <= 30:
        score += 50

    elif row["days_in_arrears"] <= 60:
        score -= 100

    else:
        score -= 200

    return score

In [20]:
#let's now apply it
df["credit_score"] = df.apply(
    credit_score,
    axis=1
)
df

,customer_id,monthly_income,days_in_arrears,z_score,risk_bucket,credit_score
0,1,50000,0,-1.214719,Current,650
1,2,70000,0,-1.057981,Current,650
2,3,85000,5,-0.940428,Watchlist,550
3,4,100000,0,-0.822874,Current,650
4,5,120000,10,-0.666136,Watchlist,550
5,6,140000,15,-0.509398,Watchlist,550
6,7,160000,0,-0.352660,Current,650
7,8,180000,20,-0.195922,Watchlist,550
8,9,200000,30,-0.039184,Watchlist,550
9,10,220000,45,0.117553,High Risk,500


In [21]:
#score to decision
def decision(score):

    if score >= 700:
        return "Approve"

    elif score >= 550:
        return "Review"

    else:
        return "Decline"

In [22]:
#apply decision
df["decision"] = df["credit_score"].apply(decision)
df

,customer_id,monthly_income,days_in_arrears,z_score,risk_bucket,credit_score,decision
0,1,50000,0,-1.214719,Current,650,Review
1,2,70000,0,-1.057981,Current,650,Review
2,3,85000,5,-0.940428,Watchlist,550,Review
3,4,100000,0,-0.822874,Current,650,Review
4,5,120000,10,-0.666136,Watchlist,550,Review
5,6,140000,15,-0.509398,Watchlist,550,Review
6,7,160000,0,-0.352660,Current,650,Review
7,8,180000,20,-0.195922,Watchlist,550,Review
8,9,200000,30,-0.039184,Watchlist,550,Review
9,10,220000,45,0.117553,High Risk,500,Decline


In [23]:
#let's now analyze the distribution of credit scores
df.groupby(
    "decision"
).agg({

    "customer_id":"count",

    "monthly_income":"mean"
})

,customer_id,monthly_income
decision,,
Approve,3,406666.666667
Decline,3,250000.000000
Review,9,122777.777778
